In [ ]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
from optuna.samplers import GPSampler
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import script_otim as script
import logging

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "gp_seed1" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_c094c75c-1ef6-493c-be80-4bd8fcb1fdca.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 61657.
PyAEDT INFO: Electronics Desktop started on gRPC port: 61657 after 18.387147426605225 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 7668.


In [ ]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    df_filtrado = df[(df["Frequency [MHz]"] >= 0.3) & (df["Frequency [MHz]"] <= 1.5)]
    s11 = np.asarray(df_filtrado['|$S_{11}$| [dB]'], dtype=float)

    n = len(s11)
    idx1 = n // 3
    idx2 = 2 * n // 3

    weight = np.zeros(n)
    weight[:idx1] = 1
    weight[idx1:idx2] = 0.9
    weight[idx2:] = 0.7

    mask = s11 >= -10

    weighted_count = np.sum(mask * weight)

    weighted_fraction = weighted_count / np.sum(weight)

    return weighted_fraction

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name,
    sampler=GPSampler(seed=123),
    load_if_exists=True
)

#seed1 = 123
#seed2 = 173
#seed3 = 247

[I 2026-01-26 09:56:55,929] Using an existing study with name 'tpe_seed2' instead of creating a new one.


Using an existing study with name 'tpe_seed2' instead of creating a new one.


In [ ]:
study.optimize(objective, n_trials=1000)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1351 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_RSB' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 3sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Deleted 1 Objects: cut.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m

[I 2026-01-26 10:15:04,979] Trial 10 finished with value: 0.9732569245463228 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 10, 'n2': 7.382935299854013, 'n3': 5.700235719492338, 'n4': 59, 'perc': 0.5638422947859616, 'r': 136}. Best is trial 1 with value: 0.8156638013371538.


Trial 10 finished with value: 0.9732569245463228 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 10, 'n2': 7.382935299854013, 'n3': 5.700235719492338, 'n4': 59, 'perc': 0.5638422947859616, 'r': 136}. Best is trial 1 with value: 0.8156638013371538.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1352 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_UG9' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Mod

[I 2026-01-26 11:34:07,593] Trial 11 finished with value: 0.5205348615090736 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 52, 'n2': 2.4567037468611463, 'n3': 1.3020888528437355, 'n4': 13, 'perc': 0.41656473177870706, 'r': 91}. Best is trial 11 with value: 0.5205348615090736.


Trial 11 finished with value: 0.5205348615090736 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 52, 'n2': 2.4567037468611463, 'n3': 1.3020888528437355, 'n4': 13, 'perc': 0.41656473177870706, 'r': 91}. Best is trial 11 with value: 0.5205348615090736.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1353 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_5NY' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D 

...

Trial 444 finished with value: 0.36485195797516723 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 44, 'n2': 8.006132366706538, 'n3': 2.286317526222522, 'n4': 21, 'perc': 0.57951457123743, 'r': 106}. Best is trial 63 with value: 0.047755491881566386.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1786 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_4AU' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D 

[I 2026-02-05 14:07:15,862] Trial 445 finished with value: 0.05730659025787966 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 43, 'n2': 7.702026459070779, 'n3': 2.003417758960507, 'n4': 18, 'perc': 0.59033728948274, 'r': 106}. Best is trial 63 with value: 0.047755491881566386.


Trial 445 finished with value: 0.05730659025787966 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 43, 'n2': 7.702026459070779, 'n3': 2.003417758960507, 'n4': 18, 'perc': 0.59033728948274, 'r': 106}. Best is trial 63 with value: 0.047755491881566386.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1787 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_HHW' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D 

[I 2026-02-05 14:41:06,082] Trial 446 finished with value: 0.047755491881566386 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 42, 'n2': 8.021056204610671, 'n3': 2.268870812445842, 'n4': 19, 'perc': 0.5720107480722477, 'r': 104}. Best is trial 63 with value: 0.047755491881566386.


Trial 446 finished with value: 0.047755491881566386 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 42, 'n2': 8.021056204610671, 'n3': 2.268870812445842, 'n4': 19, 'perc': 0.5720107480722477, 'r': 104}. Best is trial 63 with value: 0.047755491881566386.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 7668!
PyAEDT INFO: Project Project1788 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_HYO' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 

In [ ]:
len(study.trials)

In [ ]:
study.best_params

In [ ]:
study.best_trial

In [ ]:
d.release_desktop()